# Paper #31 Implementation: HMI Observables Processing
# 논문 #31 구현: HMI 관측량 처리

**Paper**: Couvidat, Schou, Hoeksema et al. (2016), *Observables Processing for the Helioseismic and Magnetic Imager Instrument on the Solar Dynamics Observatory*, Solar Physics 291, 1887-1938. DOI: 10.1007/s11207-016-0957-3

## 한국어 설명
이 노트북은 HMI(Helioseismic and Magnetic Imager) 관측량 처리 파이프라인의 핵심 요소를 시뮬레이션한다. 실제 HMI 데이터 대신 합성 Fe I 6173 Å 선 프로파일과 이상적 필터 투과 함수를 사용해 다음을 재현한다:

1. Fe I 6173 Å 선의 Gaussian/Voigt 프로파일
2. MDI-like 알고리즘: 6개 파장 샘플 → Fourier 계수 → Doppler 속도, 연속 강도, 선폭, 선깊이
3. Milne-Eddington 대기에서의 Stokes I, Q, U, V 프로파일 합성
4. VFISV 스타일 벡터 자기장 인버전 데모 (강도 하강법)
5. 간단한 180° 방향 모호성 해소
6. 플랫필드 보정 시뮬레이션

## English description
This notebook simulates the key steps of HMI's observables pipeline. Instead of real HMI data, we use synthetic Fe I 6173 Å line profiles and idealized filter transmittances to reproduce:

1. The Fe I 6173 Å line Gaussian/Voigt profile
2. The MDI-like algorithm: 6 wavelength samples → Fourier coefficients → Doppler velocity, continuum, line width, line depth
3. Stokes I, Q, U, V synthesis from a Milne-Eddington atmosphere
4. A VFISV-style vector-field inversion demo (gradient descent)
5. A toy 180° azimuth ambiguity resolver
6. Flat-field correction simulation

In [ ]:
"""Standard imports for HMI observables simulation."""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import wofz

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

## 1. Fe I 6173 Å Line Profile / Fe I 6173 Å 선 프로파일

### 한국어
HMI는 Fe I 6173.3433 Å 흡수선을 관측한다. 본 절에서는 Gaussian 프로파일과 Voigt 프로파일을 함께 시각화해, MDI-like 알고리즘이 가정하는 Gaussian 모델과 실제 태양 선 형태의 차이를 보여준다.

### English
HMI observes the Fe I 6173.3433 Å absorption line. We visualize a Gaussian profile alongside a Voigt profile to illustrate the difference between the MDI-like algorithm's Gaussian assumption and the real solar line shape.

In [ ]:
# Fe I line parameters (reference values from Couvidat+ 2016).
LAMBDA0 = 6173.3433  # Line center wavelength [Å].
C_LIGHT = 299792458.0  # Speed of light [m/s].
DVDL = C_LIGHT / LAMBDA0  # Wavelength-to-velocity conversion [m/s/Å] ≈ 48562.4.
LANDE_G = 2.5  # Landé factor for Fe I 6173.


def gaussian_line(wavelength, ic=1.0, id_depth=0.62, sigma=0.0613, lam0=LAMBDA0):
    """Synthesize a Gaussian Fe I line profile.

    Args:
        wavelength: Array of wavelengths [Å].
        ic: Continuum intensity (arbitrary units).
        id_depth: Line depth (fraction of continuum).
        sigma: Gaussian width [Å]; 0.0613 Å gives FWHM ~102 mÅ matching HMI.
        lam0: Line-center wavelength [Å].

    Returns:
        Array of intensities matching ``wavelength``.
    """
    return ic - id_depth * np.exp(-((wavelength - lam0) ** 2) / sigma**2)


def voigt_line(wavelength, ic=1.0, id_depth=0.62, sigma=0.04, gamma=0.015, lam0=LAMBDA0):
    """Synthesize a Voigt Fe I line profile via the Faddeeva function.

    Args:
        wavelength: Array of wavelengths [Å].
        ic: Continuum intensity.
        id_depth: Peak absorption depth.
        sigma: Gaussian standard deviation [Å].
        gamma: Lorentzian HWHM [Å].
        lam0: Line-center wavelength [Å].

    Returns:
        Array of intensities matching ``wavelength``.
    """
    z = ((wavelength - lam0) + 1j * gamma) / (sigma * np.sqrt(2.0))
    voigt = np.real(wofz(z)) / (sigma * np.sqrt(2 * np.pi))
    voigt /= voigt.max()  # Normalize to unit peak.
    return ic - id_depth * voigt

In [ ]:
# HMI samples six wavelengths separated by 68.8 mÅ, total span 412.8 mÅ.
WAVELENGTH_STEP = 0.0688  # [Å]
N_SAMPLES = 6
lam_samples = LAMBDA0 + (np.arange(N_SAMPLES) - (N_SAMPLES - 1) / 2) * WAVELENGTH_STEP

lam_dense = np.linspace(LAMBDA0 - 0.4, LAMBDA0 + 0.4, 801)
profile_gauss = gaussian_line(lam_dense)
profile_voigt = voigt_line(lam_dense)

fig, ax = plt.subplots()
ax.plot(lam_dense, profile_gauss, label="Gaussian (MDI-like assumption)")
ax.plot(lam_dense, profile_voigt, "--", label="Voigt (realistic)")
ax.scatter(lam_samples, gaussian_line(lam_samples), color="red", zorder=3,
           label="HMI 6 samples")
ax.axvline(LAMBDA0, color="gray", linewidth=0.5)
ax.set_xlabel("Wavelength [Å]")
ax.set_ylabel("Intensity (normalized)")
ax.set_title("Fe I 6173 Å line: Gaussian vs Voigt profile with HMI's 6-sample grid\n"
             "Fe I 6173 Å 선: Gaussian vs Voigt 프로파일과 HMI 6 샘플")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 2. MDI-like Algorithm / MDI 유사 알고리즘

### 한국어
MDI-like 알고리즘은 6개 파장 샘플 $I_j$ ($j=0, \ldots, 5$)로부터 이산 Fourier 계수를 계산한다 (식 11):

$$a_n \approx \frac{2}{6}\sum_{j=0}^{5} I_j \cos\!\left(2\pi n \frac{2.5-j}{6}\right), \qquad b_n \approx \frac{2}{6}\sum_{j=0}^{5} I_j \sin\!\left(2\pi n \frac{2.5-j}{6}\right).$$

이로부터 도플러 속도 (식 7), 선폭 (식 10), 선깊이 (식 9), 연속 강도 (식 14)를 유도한다. 주기 $T = 6 \times 68.8 = 412.8$ mÅ.

### English
The MDI-like algorithm computes discrete Fourier coefficients from 6 wavelength samples $I_j$ ($j = 0, \ldots, 5$) per Eq. (11). The period is $T = 6 \times 68.8 = 412.8$ mÅ. Doppler velocity, line width, line depth, and reconstructed continuum follow from Eqs. (7), (10), (9), (14).

In [ ]:
T_SPAN = N_SAMPLES * WAVELENGTH_STEP  # 0.4128 Å = 412.8 mÅ.


def mdi_like_fourier(intensities):
    """Compute discrete first and second Fourier coefficients (Eq. 11).

    Args:
        intensities: Array of 6 filtergram intensities.

    Returns:
        Tuple (a1, b1, a2, b2).
    """
    j = np.arange(N_SAMPLES)
    phase1 = 2 * np.pi * (2.5 - j) / N_SAMPLES
    phase2 = 4 * np.pi * (2.5 - j) / N_SAMPLES
    norm = 2.0 / N_SAMPLES
    a1 = norm * np.sum(intensities * np.cos(phase1))
    b1 = norm * np.sum(intensities * np.sin(phase1))
    a2 = norm * np.sum(intensities * np.cos(phase2))
    b2 = norm * np.sum(intensities * np.sin(phase2))
    return a1, b1, a2, b2


def mdi_like_observables(intensities):
    """Derive Doppler velocity, line width, depth, and continuum.

    Args:
        intensities: Array of 6 filtergram intensities.

    Returns:
        Dict with keys ``velocity`` [m/s], ``sigma`` [Å], ``depth``, ``continuum``.
    """
    a1, b1, a2, b2 = mdi_like_fourier(intensities)
    # Eq. (7): Doppler velocity.
    v = DVDL * (T_SPAN / (2 * np.pi)) * np.arctan2(b1, a1)
    # Eq. (10): Gaussian sigma from Fourier ratio.
    ratio = (a1**2 + b1**2) / max(a2**2 + b2**2, 1e-20)
    sigma = (T_SPAN / (np.pi * np.sqrt(6))) * np.sqrt(np.log(ratio))
    # Eq. (9): line depth (apply empirical K_1=5/6 correction).
    depth = (T_SPAN / (2 * sigma * np.sqrt(np.pi))) * np.sqrt(a1**2 + b1**2) * \
            np.exp(np.pi**2 * sigma**2 / T_SPAN**2)
    depth *= 5.0 / 6.0
    sigma *= 6.0 / 5.0
    # Eq. (14): reconstructed continuum.
    continuum = np.mean(intensities + depth * np.exp(-((lam_samples - LAMBDA0) ** 2) / sigma**2))
    return {"velocity": v, "sigma": sigma, "depth": depth, "continuum": continuum}

In [ ]:
# Test: inject a +500 m/s Doppler shift and recover it.
input_velocity = 500.0  # m/s.
shifted_lam0 = LAMBDA0 * (1 + input_velocity / C_LIGHT)
I_j = gaussian_line(lam_samples, lam0=shifted_lam0)
obs = mdi_like_observables(I_j)
print(f"Input velocity     : {input_velocity:.2f} m/s")
print(f"Recovered velocity : {obs['velocity']:.2f} m/s")
print(f"Line width sigma   : {obs['sigma']*1000:.2f} mÅ (input 61.3 mÅ)")
print(f"Line depth         : {obs['depth']:.3f} (input 0.620)")
print(f"Continuum          : {obs['continuum']:.3f} (input 1.000)")

In [ ]:
# Scan input velocities to reproduce the LUT shape of Fig. 12.
v_in = np.linspace(-6000, 6000, 121)
v_out = np.zeros_like(v_in)
for idx, v in enumerate(v_in):
    shifted = LAMBDA0 * (1 + v / C_LIGHT)
    v_out[idx] = mdi_like_observables(gaussian_line(lam_samples, lam0=shifted))["velocity"]

fig, ax = plt.subplots()
ax.plot(v_in, v_out, label="Raw MDI-like (no LUT)")
ax.plot(v_in, v_in, "--", color="gray", label="Ideal y = x")
ax.set_xlabel("Input solar velocity [m/s]")
ax.set_ylabel("Output Fourier velocity [m/s]")
ax.set_title("LUT shape: input vs output velocity / 룩업 테이블 형태")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 3. LoS Magnetic Field from LCP/RCP / LCP·RCP로부터 시선방향 자기장

### 한국어
Zeeman 효과로 자기장이 $B$일 때 LCP는 $+\Delta \lambda_B$, RCP는 $-\Delta \lambda_B$만큼 파장 이동한다 (원형편광 성분). MDI-like 알고리즘을 두 원편광에 각각 적용해 $V_\mathrm{LCP}, V_\mathrm{RCP}$를 얻고, 식 (12), (13)으로부터 속도와 자기장을 유도한다:

$$V = \tfrac{1}{2}(V_\mathrm{LCP} + V_\mathrm{RCP}), \qquad B = K_m(V_\mathrm{LCP} - V_\mathrm{RCP}),$$

여기서 $K_m = 0.231$ G m$^{-1}$ s ($g_L = 2.5$).

### English
In the Zeeman regime the LCP and RCP profiles shift by $\pm \Delta \lambda_B$. Applying MDI-like analysis to each circular polarization gives $V_\mathrm{LCP}, V_\mathrm{RCP}$. Equations (12) and (13) give

$$V = \tfrac{1}{2}(V_\mathrm{LCP} + V_\mathrm{RCP}), \quad B = K_m(V_\mathrm{LCP} - V_\mathrm{RCP}),$$

with $K_m = 0.231$ G m$^{-1}$ s ($g_L = 2.5$).

In [ ]:
K_M = 1.0 / (2.0 * 4.67e-5 * LAMBDA0 * LANDE_G * C_LIGHT)
print(f"K_m = {K_M:.4e} G m^-1 s (expected ~0.231)")


def zeeman_shift(b_gauss, lande_g=LANDE_G, lam0=LAMBDA0):
    """Compute the Zeeman wavelength shift for a given LoS magnetic field.

    Args:
        b_gauss: LoS magnetic field strength [G].
        lande_g: Landé factor (default 2.5 for Fe I 6173).
        lam0: Rest wavelength [Å].

    Returns:
        Wavelength shift Δλ_B [Å].
    """
    return 4.67e-5 * lande_g * (lam0**2) * b_gauss * 1e-8  # Å (classical Zeeman).


def synthesize_lcp_rcp(v_bulk, b_los):
    """Synthesize 6-sample LCP and RCP intensities.

    Args:
        v_bulk: Bulk Doppler velocity [m/s].
        b_los: LoS magnetic field [G].

    Returns:
        Tuple (I_LCP, I_RCP) each of shape (6,).
    """
    shift_doppler = LAMBDA0 * v_bulk / C_LIGHT
    shift_zeeman = zeeman_shift(b_los)
    lam_lcp = LAMBDA0 + shift_doppler + shift_zeeman
    lam_rcp = LAMBDA0 + shift_doppler - shift_zeeman
    return gaussian_line(lam_samples, lam0=lam_lcp), gaussian_line(lam_samples, lam0=lam_rcp)


# Test: V = 0, B_LoS = 100 G.
I_lcp, I_rcp = synthesize_lcp_rcp(v_bulk=0.0, b_los=100.0)
v_lcp = mdi_like_observables(I_lcp)["velocity"]
v_rcp = mdi_like_observables(I_rcp)["velocity"]
V = 0.5 * (v_lcp + v_rcp)
B = K_M * (v_lcp - v_rcp)
print(f"V_LCP = {v_lcp:.2f} m/s")
print(f"V_RCP = {v_rcp:.2f} m/s")
print(f"Recovered V      = {V:.2f} m/s (input 0)")
print(f"Recovered B_LoS  = {B:.2f} G (input 100 G)")

In [ ]:
# Sweep input B_LoS to show linearity and eventual saturation in strong fields.
b_inputs = np.linspace(-3000, 3000, 61)
b_recovered = np.zeros_like(b_inputs)
for idx, b in enumerate(b_inputs):
    I_lcp, I_rcp = synthesize_lcp_rcp(v_bulk=0.0, b_los=b)
    v_l = mdi_like_observables(I_lcp)["velocity"]
    v_r = mdi_like_observables(I_rcp)["velocity"]
    b_recovered[idx] = K_M * (v_l - v_r)

fig, ax = plt.subplots()
ax.plot(b_inputs, b_recovered, "o-", label="MDI-like retrieval")
ax.plot(b_inputs, b_inputs, "--", color="gray", label="Ideal")
ax.set_xlabel("Input B_LoS [G]")
ax.set_ylabel("Retrieved B_LoS [G]")
ax.set_title("MDI-like LoS field recovery vs input / MDI-like LoS 자기장 복원")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 4. Milne-Eddington Stokes Synthesis / Milne-Eddington Stokes 합성

### 한국어
VFISV가 반전하는 Stokes I, Q, U, V 프로파일을 Milne-Eddington 대기에서 합성한다. ME 대기 가정: 흡수계수와 자기장은 광학 깊이와 무관하며, 광원 함수 $S = S_0 + S_1 \tau$만 선형. HMI 벡터 파이프라인은 6 파장 x 4 편광 = 24 이미지를 입력으로 사용한다.

### English
Synthesize the Stokes I, Q, U, V profiles in a Milne-Eddington atmosphere - the same input VFISV inverts. The ME assumption: absorption coefficient and magnetic field are independent of optical depth; only the source function $S = S_0 + S_1 \tau$ is linear. HMI's vector pipeline uses 6 wavelengths x 4 polarizations = 24 images as input.

In [ ]:
def me_stokes(wavelengths, b_field, inclination, azimuth, v_los=0.0,
              eta0=10.0, damping=0.05, doppler_width=0.03,
              s0=0.2, s1=0.8, lam0=LAMBDA0):
    """Compute Stokes [I, Q, U, V] for a Milne-Eddington atmosphere.

    This is a pedagogical implementation following Landi degl'Innocenti &
    Landolfi (2004), simplified to the Gaussian-core limit.

    Args:
        wavelengths: Wavelength grid [Å].
        b_field: Magnetic field magnitude [G].
        inclination: Inclination angle to LoS [rad].
        azimuth: Azimuth angle [rad].
        v_los: LoS Doppler velocity [m/s].
        eta0: Line-to-continuum opacity ratio.
        damping: Damping parameter a.
        doppler_width: Doppler width [Å].
        s0: Source function constant term.
        s1: Source function slope term.
        lam0: Line-center wavelength [Å].

    Returns:
        Tuple of arrays (I, Q, U, V) with the same length as ``wavelengths``.
    """
    shift = lam0 * v_los / C_LIGHT
    zeeman = zeeman_shift(b_field)

    def profile(lam_shift):
        u = (wavelengths - lam0 - shift - lam_shift) / doppler_width
        h = np.real(wofz(u + 1j * damping))
        return h

    phi_p = profile(+zeeman)
    phi_m = profile(-zeeman)
    phi_0 = profile(0.0)

    # Absorption matrix elements.
    eta_i = 1.0 + 0.5 * eta0 * (phi_0 * np.sin(inclination) ** 2 +
                                 0.5 * (phi_p + phi_m) * (1 + np.cos(inclination) ** 2))
    eta_q = 0.5 * eta0 * (phi_0 - 0.5 * (phi_p + phi_m)) * np.sin(inclination) ** 2 * np.cos(2 * azimuth)
    eta_u = 0.5 * eta0 * (phi_0 - 0.5 * (phi_p + phi_m)) * np.sin(inclination) ** 2 * np.sin(2 * azimuth)
    eta_v = 0.5 * eta0 * (phi_p - phi_m) * np.cos(inclination)

    # Unno-Rachkovsky solution (simplified).
    denom = eta_i**2
    I = s0 + s1 * eta_i / denom
    Q = -s1 * eta_q / denom
    U = -s1 * eta_u / denom
    V = -s1 * eta_v / denom
    return I, Q, U, V

In [ ]:
# Synthesize Stokes profiles for a sunspot-like strong field with inclination = 60°.
lam_dense = np.linspace(LAMBDA0 - 0.3, LAMBDA0 + 0.3, 201)
I_d, Q_d, U_d, V_d = me_stokes(lam_dense, b_field=1500.0,
                                inclination=np.deg2rad(60.0),
                                azimuth=np.deg2rad(30.0))
I_s, Q_s, U_s, V_s = me_stokes(lam_samples, b_field=1500.0,
                                inclination=np.deg2rad(60.0),
                                azimuth=np.deg2rad(30.0))

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (prof_d, prof_s, label) in zip(axes.flat,
                                       [(I_d, I_s, "Stokes I"),
                                        (Q_d, Q_s, "Stokes Q"),
                                        (U_d, U_s, "Stokes U"),
                                        (V_d, V_s, "Stokes V")]):
    ax.plot(lam_dense, prof_d)
    ax.scatter(lam_samples, prof_s, color="red", s=30, zorder=3, label="6 samples")
    ax.axvline(LAMBDA0, color="gray", linewidth=0.5)
    ax.set_xlabel("Wavelength [Å]")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=9)
fig.suptitle("Synthetic HMI Stokes profiles for B=1500 G, inclination=60°\n"
             "HMI Stokes 합성 프로파일 (B=1500 G, 기울기=60°)")
fig.tight_layout()
plt.show()

## 5. VFISV-like Vector Field Inversion Demo / VFISV 유사 벡터 자기장 반전 데모

### 한국어
VFISV는 Milne-Eddington 모델을 6개 파장의 Stokes [I, Q, U, V] (24 관측값)에 대해 Levenberg-Marquardt 방법으로 맞춘다. 여기서는 단순화를 위해 (B, inclination, azimuth)만 미지수로 두고 scipy.optimize.minimize로 가장 기초적인 인버전을 수행한다.

### English
VFISV fits a Milne-Eddington model to the 6-wavelength Stokes [I, Q, U, V] (24 measurements) using Levenberg-Marquardt. For pedagogy we fit only (B, inclination, azimuth) using scipy.optimize.minimize.

In [ ]:
def vfisv_loss(params, wavelengths, stokes_obs, noise=0.001):
    """Chi-squared loss for a 3-parameter Milne-Eddington inversion.

    Args:
        params: Array [B, inclination_rad, azimuth_rad].
        wavelengths: Wavelength grid of the observation [Å].
        stokes_obs: Array shape (4, N_wl) with observed Stokes [I, Q, U, V].
        noise: Assumed measurement uncertainty (fraction of continuum).

    Returns:
        Chi-squared scalar.
    """
    b, inc, az = params
    I_m, Q_m, U_m, V_m = me_stokes(wavelengths, b_field=b, inclination=inc, azimuth=az)
    model = np.stack([I_m, Q_m, U_m, V_m])
    return np.sum(((model - stokes_obs) / noise) ** 2)


def vfisv_invert(wavelengths, stokes_obs):
    """Simplified VFISV-style Milne-Eddington inversion.

    Args:
        wavelengths: Wavelength grid [Å].
        stokes_obs: (4, N_wl) observed Stokes vector.

    Returns:
        Dict with inferred ``B``, ``inclination``, ``azimuth``.
    """
    x0 = np.array([500.0, np.pi / 4, np.pi / 4])
    bounds = [(0.0, 5000.0), (0.0, np.pi), (0.0, np.pi)]
    result = minimize(vfisv_loss, x0, args=(wavelengths, stokes_obs),
                      method="L-BFGS-B", bounds=bounds)
    b, inc, az = result.x
    return {"B": b, "inclination": np.rad2deg(inc), "azimuth": np.rad2deg(az),
            "success": result.success, "chi2": result.fun}


# Generate synthetic observation with known parameters.
B_TRUE = 1200.0
INC_TRUE = np.deg2rad(55.0)
AZ_TRUE = np.deg2rad(40.0)
I_s, Q_s, U_s, V_s = me_stokes(lam_samples, b_field=B_TRUE,
                                inclination=INC_TRUE, azimuth=AZ_TRUE)
noise_level = 1e-3
observed = np.stack([I_s, Q_s, U_s, V_s]) + noise_level * np.random.randn(4, N_SAMPLES)

result = vfisv_invert(lam_samples, observed)
print(f"True  : B = {B_TRUE:.1f} G, inclination = {np.rad2deg(INC_TRUE):.1f}°, "
      f"azimuth = {np.rad2deg(AZ_TRUE):.1f}°")
print(f"VFISV : B = {result['B']:.1f} G, inclination = {result['inclination']:.1f}°, "
      f"azimuth = {result['azimuth']:.1f}°")
print(f"chi^2 = {result['chi2']:.2f}, success = {result['success']}")

## 6. 180° Azimuth Ambiguity Resolution / 180° 방위각 모호성 해소

### 한국어
Stokes Q, U는 $\cos 2\phi, \sin 2\phi$로 변하므로 $\phi$와 $\phi + 180°$를 구분할 수 없다. 가장 간단한 해소법은 "포텐셜(무전류) 자기장 근접" 기준으로, 관측 방위각을 수평 성분이 발산 무전류 해에 가장 가까운 쪽으로 선택한다. 여기서는 더 교육적인 "최소 발산" 토이 알고리즘을 사용한다.

### English
Stokes Q, U depend on $\cos 2\phi, \sin 2\phi$, so $\phi$ and $\phi + 180°$ are indistinguishable. A common resolver picks the branch minimizing the divergence of the horizontal field (potential-field acute-angle method). Below, we use a didactic "minimum divergence" toy algorithm on a 2D field map.

In [ ]:
def resolve_180_ambiguity(bx_pair, by_pair):
    """Resolve the 180° ambiguity by picking the branch with the lowest divergence.

    Args:
        bx_pair: 2D array of horizontal-x field (ambiguous up to sign).
        by_pair: 2D array of horizontal-y field (ambiguous up to sign).

    Returns:
        (bx_resolved, by_resolved) with sign chosen to minimize divergence.
    """
    # Candidate A: sign kept as-is; Candidate B: sign flipped (phi + 180°).
    def divergence(bx, by):
        dbx_dx = np.gradient(bx, axis=1)
        dby_dy = np.gradient(by, axis=0)
        return dbx_dx + dby_dy

    div_a = np.sum(divergence(bx_pair, by_pair) ** 2)
    div_b = np.sum(divergence(-bx_pair, -by_pair) ** 2)
    if div_b < div_a:
        return -bx_pair, -by_pair
    return bx_pair, by_pair


# Generate a smooth dipole-like horizontal field, then flip random tiles.
ny, nx = 32, 32
yy, xx = np.mgrid[-1:1:ny * 1j, -1:1:nx * 1j]
bx_true = -yy / (xx**2 + yy**2 + 0.1)
by_true = xx / (xx**2 + yy**2 + 0.1)

mask = np.random.rand(ny, nx) > 0.5
bx_obs = np.where(mask, -bx_true, bx_true)
by_obs = np.where(mask, -by_true, by_true)

bx_resolved, by_resolved = resolve_180_ambiguity(bx_obs, by_obs)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (bx, by, title) in zip(axes,
                               [(bx_true, by_true, "True field / 참값"),
                                (bx_obs, by_obs, "Ambiguous / 모호한 관측값"),
                                (bx_resolved, by_resolved, "Resolved (toy) / 해소값")]):
    ax.quiver(xx, yy, bx, by, scale=50)
    ax.set_aspect("equal")
    ax.set_title(title)
fig.suptitle("180° azimuth ambiguity (global-sign) / 180° 방위각 모호성 (전역 부호)")
plt.tight_layout()
plt.show()

## 7. Flat-Field Correction / 플랫필드 보정

### 한국어
HMI Level-1 처리의 핵심 단계 중 하나인 플랫필드 보정을 시뮬레이션한다. CCD 응답이 픽셀마다 다를 때, 알려진 균일 광원(또는 태양 자전을 이용한 간접 측정) 이미지를 사용해 $I_\mathrm{cal} = I_\mathrm{raw} / F$로 보정한다. 여기서 $F$는 플랫필드 이미지(평균을 1로 정규화).

### English
Simulate the flat-field correction, one of HMI's core Level-1 steps. When the CCD response varies pixel-to-pixel, a known uniform source (or, in HMI's case, an inferred flat from solar-rotation data per Wachter & Schou 2009) gives $I_\mathrm{cal} = I_\mathrm{raw} / F$, with $F$ normalized to mean 1.

In [ ]:
def simulate_flatfield(shape, amplitude=0.05, seed=0):
    """Create a smooth multiplicative flat-field with a few vignetting and speckle features.

    Args:
        shape: (ny, nx) tuple.
        amplitude: RMS amplitude of deviations from unity.
        seed: Random seed.

    Returns:
        Array with mean 1.0 and the requested RMS fluctuation.
    """
    rng = np.random.default_rng(seed)
    ny, nx = shape
    yy, xx = np.mgrid[-1:1:ny * 1j, -1:1:nx * 1j]
    vignette = 1.0 - 0.08 * (xx**2 + yy**2)
    speckle = rng.normal(scale=amplitude, size=shape)
    flat = vignette + speckle
    return flat / flat.mean()


ny, nx = 256, 256
flat = simulate_flatfield((ny, nx), amplitude=0.04, seed=1)

# Synthetic "true" solar scene with limb darkening.
yy, xx = np.mgrid[-1:1:ny * 1j, -1:1:nx * 1j]
r = np.sqrt(xx**2 + yy**2)
mu = np.sqrt(np.clip(1 - r**2, 0.0, 1.0))
limb = 0.5 + 0.5 * mu  # Simple Eddington limb darkening.
truth = limb * (r < 0.95)
raw = truth * flat + 0.005 * np.random.randn(ny, nx)
corrected = raw / flat

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, img, title in zip(axes,
                          [flat, raw, corrected, corrected - truth],
                          ["Flat field / 플랫필드 (F)",
                           "Raw I_raw = I_true * F",
                           "Corrected I_raw / F",
                           "Residual"]):
    im = ax.imshow(img, origin="lower", cmap="viridis")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle("Flat-field correction / 플랫필드 보정 시뮬레이션")
plt.tight_layout()
plt.show()

print(f"Residual RMS after correction: {np.std((corrected - truth)[truth > 0.01]):.4f}")

## 8. Summary / 요약

### 한국어
- Fe I 6173 Å 선 프로파일을 Gaussian/Voigt 형태로 합성하고 HMI 6 파장 샘플링을 시각화했다.
- MDI-like Fourier 알고리즘으로 6 샘플 → 도플러 속도, 선폭, 선깊이, 연속 강도를 복원했다.
- LCP/RCP 분리와 Zeeman 이동으로부터 $B_\mathrm{LoS}$ 추정을 구현했으며, 강자기장 영역에서의 포화/과소추정을 확인했다.
- Milne-Eddington 대기에서 Stokes [I, Q, U, V] 프로파일을 합성하고 단순화된 VFISV 반전을 수행했다.
- 180° 방위각 모호성을 "전역 부호 + 최소 발산" 토이 알고리즘으로 해소했다.
- 플랫필드 보정의 단순 시뮬레이션을 제시했다.

### English
- Synthesized the Fe I 6173 Å line in Gaussian/Voigt form and visualized HMI's 6-wavelength sampling.
- Implemented the MDI-like Fourier algorithm recovering Doppler velocity, line width, depth, and continuum from 6 samples.
- Estimated $B_\mathrm{LoS}$ from LCP/RCP split via the Zeeman effect, demonstrating saturation in strong fields.
- Synthesized Stokes [I, Q, U, V] in a Milne-Eddington atmosphere and ran a simplified VFISV inversion.
- Resolved the 180° azimuth ambiguity with a toy "global-sign + minimum-divergence" algorithm.
- Demonstrated flat-field correction on a synthetic limb-darkened image.